# Run Distributed PyTorch Training with JobSet

Use a `JobSet` custom resource to run a small two-node PyTorch Distributed Data Parallel (DDP) training job. JobSet creates and manages the Kubernetes Jobs and stable pod DNS names; `torchrun` starts the distributed PyTorch processes.

This example trains a linear model on synthetic CPU data. It uses one indexed Kubernetes Job with two pods, so each pod gets a stable completion index that `torchrun` uses as its node rank. The training code is mounted from a ConfigMap and needs no shared PVC.

Use JobSet directly when you want to define the Kubernetes job and pod templates yourself. If you want reusable framework runtimes and a higher-level training API, use [Kubeflow Trainer v2](./fine-tune-with-trainer-v2.ipynb); Trainer creates JobSets for its `TrainJob` resources.

## Prerequisites

- The Alauda Build of JobSet controller is installed and running. See [Install JobSet](../components/jobset/install.mdx).
- The `jobset.x-k8s.io/v1alpha2` API is available and you can create `JobSet` resources in your namespace.
- Two schedulable CPU nodes are available.
- The namespace can pull `alaudadockerhub/torch-distributed:v2.9.1-aml2`, or the image field in the manifest is changed to an accessible image with PyTorch 2.9.1, `torchrun`, and Bash.

## Set the namespace

Run this notebook from `docs/en/train/guides` in an `aml-docs` checkout. The training script and JobSet manifest are in the adjacent `assets/` directory.

In [ ]:
import os

NAMESPACE = os.environ.get("NAMESPACE", "finetune")
print(f"Using namespace: {NAMESPACE}")

## Create the training code ConfigMap

Create or update the ConfigMap from the sample script:

In [ ]:
!kubectl create configmap torch-ddp-training --from-file=train.py=assets/jobset-pytorch-ddp/train.py --namespace {NAMESPACE}


## Submit the JobSet

Apply the JobSet manifest. It creates one indexed Job with two pods. JobSet publishes their stable hostnames before the pods are ready, which lets rank 1 resolve rank 0 while `torchrun` is starting.

In [ ]:
!kubectl apply --namespace {NAMESPACE} -f assets/jobset-pytorch-ddp/jobset.yaml
!kubectl wait --for=condition=Completed jobset/torch-ddp --namespace {NAMESPACE} --timeout=10m


## Check the training run

List the JobSet, child Job, pods, and headless Service:

In [ ]:
!kubectl get jobset torch-ddp --namespace {NAMESPACE}
!kubectl get jobs,pods,services --namespace {NAMESPACE} \
  -l jobset.sigs.k8s.io/jobset-name=torch-ddp

Read logs from both training ranks. The logs should contain rank `0/2` and `1/2` with similar model weights and biases, showing that DDP synchronized the model parameters.

In [ ]:
!kubectl logs --namespace {NAMESPACE} -l jobset.sigs.k8s.io/jobset-name=torch-ddp -c trainer --prefix=true


## Clean up

Delete the JobSet and ConfigMap when the run is complete:

In [ ]:
!kubectl delete jobset torch-ddp --namespace {NAMESPACE}
!kubectl delete configmap torch-ddp-training --namespace {NAMESPACE}

For more JobSet options, see the upstream [JobSet concepts guide](https://jobset.sigs.k8s.io/docs/concepts/) and the platform's [JobSet quickstart](../components/jobset/quickstart.mdx).